In [22]:
from bokeh.plotting import figure, show
from bokeh.io import output_file, output_notebook
from bokeh.layouts import row, column
from bokeh.models import ColumnDataSource, CustomJS
from bokeh.models.widgets import Slider, TextInput
import numpy as np

output_notebook()
output_file("sine_wave.html")

N = 200
x = np.linspace(0, 4*np.pi, N)
y = np.sin(x)
source = ColumnDataSource(data=dict(x=x, y=y))

# plot
plot = figure(height=400, width=600, title="my sine wave",
            tools="crosshair,pan,reset,save,wheel_zoom",
            x_range=[0, 4*np.pi], y_range=[-2.5, 2.5])

plot.line('x', 'y', source=source, line_width=3, line_alpha=0.6)

# weidgets
text = TextInput(title="title", value='my sine wave')
offset = Slider(title="offset", value=0.0, start=-5.0, end=5.0, step=0.1)
amplitude = Slider(title="amplitude", value=1.0, start=-5.0, end=5.0, step=0.1)
phase = Slider(title="phase", value=0.0, start=0.0, end=2*np.pi)
freq = Slider(title="frequency", value=1.0, start=0.1, end=5.1, step=0.1)

# CUSTOM JS CALLBACKS FFS
callback = CustomJS(args=dict(source=source, amp=amplitude, offset=offset, phase=phase, freq=freq), code="""
    const data = source.data;
    const A = amp.value;
    const B = offset.value;
    const w = phase.value;
    const k = freq.value;
    
    const x = data['x'];
    const y = data['y'];
    
    for (let i = 0; i < x.length; i++) {
        y[i] = A * Math.sin(k * x[i] + w) + B;
    }
    
    source.change.emit();
""")

title_callback = CustomJS(args=dict(plot=plot), code="""
    plot.title.text = cb_obj.value;
""")

# connecting callbacks
amplitude.js_on_change('value', callback)
offset.js_on_change('value', callback)
phase.js_on_change('value', callback)
freq.js_on_change('value', callback)
text.js_on_change('value', title_callback)

# layout
layout = column(
    plot,
    text,
    offset,
    amplitude,
    phase,
    freq
)

# the resulting plot
show(layout)

Loading BokehJS ...

In [25]:
import panel as pn
import numpy as np
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
## https://justinbois.github.io/bootcamp/2021/lessons/l30_javascript_for_bokeh.html
pn.extension()

# Create interactive function with Panel
@pn.depends(amplitude=pn.widgets.FloatSlider(value=1, start=-5, end=5, step=0.1),
            offset=pn.widgets.FloatSlider(value=0, start=-5, end=5, step=0.1),
            phase=pn.widgets.FloatSlider(value=0, start=0, end=2*np.pi, step=0.1),
            freq=pn.widgets.FloatSlider(value=1, start=0.1, end=5, step=0.1))
def sine_plot(amplitude, offset, phase, freq):
    N = 200
    x = np.linspace(0, 4*np.pi, N)
    y = amplitude * np.sin(freq * x + phase) + offset
    
    source = ColumnDataSource(data=dict(x=x, y=y))
    
    p = figure(height=400, width=600, title="Interactive Sine Wave",
              tools="crosshair,pan,reset,save,wheel_zoom",
              x_range=[0, 4*np.pi], y_range=[-5, 5])
    p.line('x', 'y', source=source, line_width=3, line_alpha=0.6)
    return p

# Display the app
pn.Column(sine_plot).servable()

Column
    [0] ParamFunction(function, _pane=Bokeh, defer_load=False)

In [28]:
from bokeh.plotting import figure, show, output_file, output_notebook
from bokeh.models import ColumnDataSource, CustomJS
from bokeh.models.widgets import Slider, TextInput
from bokeh.layouts import column
import numpy as np

def auto_js_callback(source, param_widgets, formula_js):
    """
    Generate a CustomJS callback that applies a given JS formula to update the data source.

    Args:
        source (ColumnDataSource): The Bokeh data source.
        param_widgets (dict): Dictionary mapping parameter names to widget instances.
        formula_js (str): A JavaScript expression string that uses 'x[i]' and the parameters.
                          For example: "amp * Math.sin(freq * x[i] + phase) + offset"
    Returns:
        CustomJS: A Bokeh CustomJS callback instance.
    """
    # Dynamically construct variable declarations from the widget parameters.
    code_param_lines = ""
    for param in param_widgets.keys():
        code_param_lines += f"const {param} = {param}.value;\n"

    # Create the full callback code
    custom_js_code = f"""
    const data = source.data;
    const x = data['x'];
    const y = data['y'];
    {code_param_lines}
    for (let i = 0; i < x.length; i++) {{
        y[i] = {formula_js};
    }}
    source.change.emit();
    """

    # Combine all the models to pass into the callback.
    args_dict = {'source': source}
    args_dict.update(param_widgets)
    return CustomJS(args=args_dict, code=custom_js_code)

# Configure output to both Jupyter notebook and an HTML file
output_notebook()
output_file("auto_callback.html")

# Set up some initial data (here using a sine wave as an example)
N = 200
x = np.linspace(0, 4 * np.pi, N)
y = np.sin(x)
source = ColumnDataSource(data=dict(x=x, y=y))

# Create a simple figure
plot = figure(height=400, width=600, 
              title="Auto-generated JS Callback Example",
              tools="crosshair,pan,reset,save,wheel_zoom")
plot.line('x', 'y', source=source, line_width=3, line_alpha=0.6)

# Create widget controls for the sine wave parameters
amp_slider = Slider(title="Amplitude", value=1.0, start=-5.0, end=5.0, step=0.1)
offset_slider = Slider(title="Offset", value=0.0, start=-5.0, end=5.0, step=0.1)
phase_slider = Slider(title="Phase", value=0.0, start=0.0, end=2*np.pi)
freq_slider = Slider(title="Frequency", value=1.0, start=0.1, end=5.1, step=0.1)

# Bundle the widgets with parameter names that will be used in the JS formula.
param_widgets = {
    'amp': amp_slider,
    'offset': offset_slider,
    'phase': phase_slider,
    'freq': freq_slider
}

# Define a formula in JavaScript that will be applied on each data point
# The formula uses the column data 'x' and the parameters: amp, offset, phase, and freq.
formula_js = "amp * Math.sin(freq * x[i] + phase) + offset"

# Automatically generate the callback
callback = auto_js_callback(source, param_widgets, formula_js)

# Link the generated callback to the slider widgets
for widget in param_widgets.values():
    widget.js_on_change('value', callback)

# A simple callback example for changing the plot title using a TextInput widget.
title_input = TextInput(title="Title", value="Auto-generated JS Callback Example")
title_callback = CustomJS(args=dict(plot=plot), code="plot.title.text = cb_obj.value;")
title_input.js_on_change('value', title_callback)

# Organize the layout and display the result
layout = column(plot, title_input, amp_slider, offset_slider, phase_slider, freq_slider)
show(layout)


Loading BokehJS ...

In [29]:
from bokeh.plotting import figure, show
from bokeh.io import output_file, output_notebook
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, CustomJS, Slider, TextInput
import pandas as pd
import numpy as np

# Configure output
output_notebook()
output_file("interactive_data_viz.html")

# Load data from external file
# For this example, I'll create a CSV, but you would use your actual data
sample_data = pd.DataFrame({
    'x': np.linspace(0, 4*np.pi, 200),
    'y': np.sin(np.linspace(0, 4*np.pi, 200))
})
sample_data.to_csv('sample_data.csv', index=False)

# Load the data (in a real scenario, this would be your external data)
df = pd.read_csv('sample_data.csv')

# Create a copy of the original data to preserve it
df['y_original'] = df['y'].copy()

# Set up data source
source = ColumnDataSource(df)

# Set up plot
plot = figure(height=400, width=600, title="Data Visualization",
              tools="crosshair,pan,reset,save,wheel_zoom",
              x_range=[df['x'].min(), df['x'].max()], 
              y_range=[df['y'].min()*3, df['y'].max()*3])  # Wider y-range for transformations

plot.line('x', 'y', source=source, line_width=3, line_alpha=0.6)

# Set up widgets
text = TextInput(title="Title", value='Data Visualization')
offset = Slider(title="Y Offset", value=0.0, start=-2.0, end=2.0, step=0.1)
scale = Slider(title="Y Scale", value=1.0, start=0.1, end=3.0, step=0.1)

# JavaScript callback that transforms the data rather than recreating it
callback = CustomJS(args=dict(source=source, offset=offset, scale=scale), code="""
    const data = source.data;
    const offset_val = offset.value;
    const scale_val = scale.value;
    
    const y_original = data['y_original'];
    const y = data['y'];
    
    // Apply transformations to the original data
    for (let i = 0; i < y.length; i++) {
        y[i] = y_original[i] * scale_val + offset_val;
    }
    
    source.change.emit();
""")

title_callback = CustomJS(args=dict(plot=plot), code="""
    plot.title.text = cb_obj.value;
""")

# Connect callbacks
offset.js_on_change('value', callback)
scale.js_on_change('value', callback)
text.js_on_change('value', title_callback)

# Create layout
layout = column(
    plot,
    text,
    offset,
    scale
)

# Show the result
show(layout)

Loading BokehJS ...

In [30]:
from bokeh.plotting import figure, show
from bokeh.io import output_file, output_notebook
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, CustomJS, Slider, TextInput, Range1d
import pandas as pd
import numpy as np

# Configure output
output_notebook()
output_file("pure_display_viz.html")

# Create data with pre-calculated transformations
x = np.linspace(0, 4*np.pi, 200)

# Pre-calculate ALL possible transformations
transformed_data = {}
# Original data
transformed_data['original'] = np.sin(x)

# Pre-calculate offset transformations (for -2 to 2 in 0.1 steps)
for offset in np.arange(-2.0, 2.1, 0.1):
    key = f"offset_{offset:.1f}"
    transformed_data[key] = np.sin(x) + offset

# Pre-calculate scale transformations (for 0.1 to 3.0 in 0.1 steps)
for scale in np.arange(0.1, 3.1, 0.1):
    key = f"scale_{scale:.1f}"
    transformed_data[key] = np.sin(x) * scale

# Pre-calculate combined transformations
for offset in np.arange(-2.0, 2.1, 0.1):
    for scale in np.arange(0.1, 3.1, 0.1):
        key = f"combined_{scale:.1f}_{offset:.1f}"
        transformed_data[key] = np.sin(x) * scale + offset

# Create the data source with x and current y
source = ColumnDataSource(data={
    'x': x,
    'y': transformed_data['original'],
    **transformed_data  # Include all pre-calculated data
})

# Set up plot
plot = figure(height=400, width=600, title="Data Visualization",
              tools="crosshair,pan,reset,save,wheel_zoom")
plot.y_range = Range1d(-3, 3)
plot.x_range = Range1d(0, 4*np.pi)

plot.line('x', 'y', source=source, line_width=3, line_alpha=0.6)

# Set up widgets
text = TextInput(title="Title", value='Data Visualization')
offset = Slider(title="Y Offset", value=0.0, start=-2.0, end=2.0, step=0.1)
scale = Slider(title="Y Scale", value=1.0, start=0.1, end=3.0, step=0.1)

# JavaScript callback that ONLY switches between pre-calculated datasets
callback = CustomJS(args=dict(source=source, offset=offset, scale=scale), code="""
    const data = source.data;
    const offset_val = offset.value.toFixed(1);
    const scale_val = scale.value.toFixed(1);
    
    // Look up the pre-calculated data based on current slider values
    const key = `combined_${scale_val}_${offset_val}`;
    
    // Simply replace the y values with pre-calculated values
    const y = data['y'];
    const transformed = data[key];
    
    for (let i = 0; i < y.length; i++) {
        y[i] = transformed[i];
    }
    
    source.change.emit();
""")

title_callback = CustomJS(args=dict(plot=plot), code="""
    plot.title.text = cb_obj.value;
""")

# Connect callbacks
offset.js_on_change('value', callback)
scale.js_on_change('value', callback)
text.js_on_change('value', title_callback)

# Create layout
layout = column(
    plot,
    text,
    offset,
    scale
)

# Show the result
show(layout)

Loading BokehJS ...

In [31]:
from bokeh.plotting import show
from bokeh.io import output_notebook, output_file
from bokeh.models import ColumnDataSource, DataTable, TableColumn, NumberFormatter
from bokeh.layouts import column
import pandas as pd

# Configure output
output_notebook()
output_file("bokeh_table.html")

# Create sample data
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Age': [24, 42, 33, 19, 29],
    'Department': ['Engineering', 'Marketing', 'Sales', 'Engineering', 'HR'],
    'Salary': [75000, 88000, 61000, 68000, 72000],
    'Years': [2, 10, 5, 1, 3]
}

# Create pandas DataFrame
df = pd.DataFrame(data)

# Create a Bokeh ColumnDataSource
source = ColumnDataSource(df)

# Define the columns for the DataTable
columns = [
    TableColumn(field="Name", title="Name"),
    TableColumn(field="Age", title="Age"),
    TableColumn(field="Department", title="Department"),
    TableColumn(field="Salary", title="Salary", formatter=NumberFormatter(format="$0,0")),
    TableColumn(field="Years", title="Years of Service")
]

# Create the DataTable
data_table = DataTable(
    source=source,
    columns=columns,
    width=600,
    height=280,
    index_position=None,  # Hide index column
    sortable=True,        # Enable sorting
    selectable=True,      # Enable selection
    editable=True         # Enable editing
)

# Show the table
show(data_table)

Loading BokehJS ...

In [38]:
def create_interactive_plot(data_df, 
                           x_column,
                           y_column, 
                           plot_title="Interactive Plot",
                           plot_width=800,
                           plot_height=400,
                           parameters=None,
                           callback_functions=None,
                           output_filename="interactive_plot.html"):
    """
    Creates an interactive Bokeh plot with automated JavaScript callbacks.
    
    Parameters:
    -----------
    data_df : pandas DataFrame
        The DataFrame containing the data to plot
    x_column : str
        The column name to use for x-axis
    y_column : str
        The column name to use for y-axis
    plot_title : str
        The title of the plot
    plot_width : int
        Width of the plot in pixels
    plot_height : int
        Height of the plot in pixels
    parameters : dict
        A dictionary of parameters to create sliders/inputs for.
        Format: {
            'param_name': {
                'type': 'slider' or 'text' or 'select',
                'title': 'Display Title',
                'value': initial_value,
                'start': min_value,  # For sliders
                'end': max_value,    # For sliders
                'step': step_size,   # For sliders
                'options': [list, of, options]  # For select
            }
        }
    callback_functions : dict
        A dictionary of JavaScript callback functions for each parameter.
        Format: {
            'param_name': 'JavaScript function as string that updates the data',
        }
    output_filename : str
        Filename for the HTML output
        
    Returns:
    --------
    layout : bokeh layout
        The complete Bokeh layout that can be shown with show(layout)
    """
    from bokeh.plotting import figure
    from bokeh.io import output_file
    from bokeh.layouts import column, row
    from bokeh.models import ColumnDataSource, CustomJS
    from bokeh.models.widgets import Slider, TextInput, Select
    
    # Configure output
    output_file(output_filename)
    
    # Create a copy of the original data for reference
    data_df = data_df.copy()
    for col in data_df.columns:
        data_df[f"{col}_original"] = data_df[col].copy()
    
    # Create the ColumnDataSource
    source = ColumnDataSource(data_df)
    
    # Create the figure
    plot = figure(
        title=plot_title,
        width=plot_width,
        height=plot_height,
        tools="pan,wheel_zoom,box_zoom,reset,save,crosshair"
    )
    
    # Add the main line or scatter plot
    plot.line(x_column, y_column, source=source, line_width=2)
    
    # Create widgets and callbacks
    widgets = []
    if parameters:
        # Title widget is special - always create it
        title_input = TextInput(title="Plot Title", value=plot_title)
        title_callback = CustomJS(args=dict(plot=plot), code="""
            plot.title.text = cb_obj.value;
        """)
        title_input.js_on_change('value', title_callback)
        widgets.append(title_input)
        
        # Create all the parameter widgets
        for param_name, param_config in parameters.items():
            if param_config['type'] == 'slider':
                widget = Slider(
                    title=param_config['title'],
                    value=param_config['value'],
                    start=param_config['start'],
                    end=param_config['end'],
                    step=param_config.get('step', 1)
                )
            elif param_config['type'] == 'text':
                widget = TextInput(
                    title=param_config['title'],
                    value=str(param_config['value'])
                )
            elif param_config['type'] == 'select':
                widget = Select(
                    title=param_config['title'],
                    value=str(param_config['value']),
                    options=param_config['options']
                )
            else:
                continue
                
            # Get the specific callback for this parameter or use a default
            if callback_functions and param_name in callback_functions:
                callback_code = callback_functions[param_name]
            else:
                # Default callback simply logs the change
                callback_code = f"""
                    console.log("{param_name} changed to: " + cb_obj.value);
                """
            
            # Create the callback
            callback = CustomJS(args=dict(source=source, param=widget), code=callback_code)
            widget.js_on_change('value', callback)
            widgets.append(widget)
    
    # Create and return the layout
    layout = column(plot, *widgets)
    return layout

# Example usage with automatically generated callbacks
def demo_auto_callbacks():
    import pandas as pd
    import numpy as np
    
    # Create sample data
    x = np.linspace(0, 10, 100)
    y = np.sin(x)
    data = pd.DataFrame({'x': x, 'y': y})
    
    # Define parameters
    parameters = {
        'amplitude': {
            'type': 'slider',
            'title': 'Amplitude',
            'value': 1.0,
            'start': 0.1,
            'end': 3.0,
            'step': 0.1
        },
        'frequency': {
            'type': 'slider',
            'title': 'Frequency',
            'value': 1.0,
            'start': 0.1,
            'end': 3.0,
            'step': 0.1
        },
        'offset': {
            'type': 'slider',
            'title': 'Y Offset',
            'value': 0.0,
            'start': -2.0,
            'end': 2.0,
            'step': 0.1
        },
        'line_type': {
            'type': 'select',
            'title': 'Line Type',
            'value': 'Sine',
            'options': ['Sine', 'Cosine', 'Linear']
        }
    }
    
    # Create automated callbacks
    callbacks = {
        'amplitude': '''
            const data = source.data;
            const amplitude = cb_obj.value;
            const frequency = param.document._all_models['frequency'].value;
            const offset = param.document._all_models['offset'].value;
            const line_type = param.document._all_models['line_type'].value;
            
            const x = data['x_original'];
            const y = data['y'];
            
            for (let i = 0; i < x.length; i++) {
                if (line_type === 'Sine') {
                    y[i] = amplitude * Math.sin(frequency * x[i]) + offset;
                } else if (line_type === 'Cosine') {
                    y[i] = amplitude * Math.cos(frequency * x[i]) + offset;
                } else {
                    // Linear
                    y[i] = amplitude * x[i] + offset;
                }
            }
            
            source.change.emit();
        ''',
        'frequency': '''
            const data = source.data;
            const frequency = cb_obj.value;
            const amplitude = param.document._all_models['amplitude'].value;
            const offset = param.document._all_models['offset'].value;
            const line_type = param.document._all_models['line_type'].value;
            
            const x = data['x_original'];
            const y = data['y'];
            
            for (let i = 0; i < x.length; i++) {
                if (line_type === 'Sine') {
                    y[i] = amplitude * Math.sin(frequency * x[i]) + offset;
                } else if (line_type === 'Cosine') {
                    y[i] = amplitude * Math.cos(frequency * x[i]) + offset;
                } else {
                    // Linear
                    y[i] = amplitude * x[i] + offset;
                }
            }
            
            source.change.emit();
        ''',
        'offset': '''
            const data = source.data;
            const offset = cb_obj.value;
            const amplitude = param.document._all_models['amplitude'].value;
            const frequency = param.document._all_models['frequency'].value;
            const line_type = param.document._all_models['line_type'].value;
            
            const x = data['x_original'];
            const y = data['y'];
            
            for (let i = 0; i < x.length; i++) {
                if (line_type === 'Sine') {
                    y[i] = amplitude * Math.sin(frequency * x[i]) + offset;
                } else if (line_type === 'Cosine') {
                    y[i] = amplitude * Math.cos(frequency * x[i]) + offset;
                } else {
                    // Linear
                    y[i] = amplitude * x[i] + offset;
                }
            }
            
            source.change.emit();
        ''',
        'line_type': '''
            const data = source.data;
            const line_type = cb_obj.value;
            const amplitude = param.document._all_models['amplitude'].value;
            const frequency = param.document._all_models['frequency'].value;
            const offset = param.document._all_models['offset'].value;
            
            const x = data['x_original'];
            const y = data['y'];
            
            for (let i = 0; i < x.length; i++) {
                if (line_type === 'Sine') {
                    y[i] = amplitude * Math.sin(frequency * x[i]) + offset;
                } else if (line_type === 'Cosine') {
                    y[i] = amplitude * Math.cos(frequency * x[i]) + offset;
                } else {
                    // Linear
                    y[i] = amplitude * x[i] + offset;
                }
            }
            
            source.change.emit();
        '''
    }
    
    # Create the plot
    layout = create_interactive_plot(
        data_df=data,
        x_column='x',
        y_column='y',
        plot_title='Interactive Function Plot',
        parameters=parameters,
        callback_functions=callbacks,
        output_filename='auto_callbacks_demo.html'
    )
    
    return layout

# Example usage with callback generator function
def create_callback_generator(transformation_code):
    """
    Creates a callback generator function that produces JavaScript callbacks
    based on a template and transformation code.
    
    Parameters:
    -----------
    transformation_code : str
        JavaScript code that transforms the data
        
    Returns:
    --------
    generator : function
        A function that generates callbacks for each parameter
    """
    def generator(param_name, param_list):
        """
        Generates a specific callback for a parameter.
        
        Parameters:
        -----------
        param_name : str
            The name of the parameter to create a callback for
        param_list : list
            List of all parameter names that will be used in the callback
            
        Returns:
        --------
        callback : str
            The JavaScript callback code
        """
        # Create the parameter retrieval section
        param_retrieval = []
        for p in param_list:
            if p == param_name:
                param_retrieval.append(f"const {p} = cb_obj.value;")
            else:
                param_retrieval.append(f"const {p} = param.document._all_models['{p}'].value;")
        
        param_code = '\n    '.join(param_retrieval)
        
        # Create the full callback
        callback = f"""
            const data = source.data;
            
            // Get all parameters
            {param_code}
            
            // Get data arrays
            const x = data['x_original'];
            const y = data['y'];
            
            // Apply transformation
            {transformation_code}
            
            // Notify of changes
            source.change.emit();
        """
        
        return callback
    
    return generator

# Example of using the callback generator
def demo_callback_generator():
    import pandas as pd
    import numpy as np
    
    # Create sample data
    x = np.linspace(0, 10, 100)
    y = np.sin(x)
    data = pd.DataFrame({'x': x, 'y': y})
    
    # Define parameters
    parameters = {
        'amplitude': {
            'type': 'slider',
            'title': 'Amplitude',
            'value': 1.0,
            'start': 0.1,
            'end': 3.0,
            'step': 0.1
        },
        'frequency': {
            'type': 'slider',
            'title': 'Frequency',
            'value': 1.0,
            'start': 0.1,
            'end': 3.0,
            'step': 0.1
        },
        'offset': {
            'type': 'slider',
            'title': 'Y Offset',
            'value': 0.0,
            'start': -2.0,
            'end': 2.0,
            'step': 0.1
        }
    }
    
    # Create a transformation template
    transformation_code = """
        for (let i = 0; i < x.length; i++) {
            y[i] = amplitude * Math.sin(frequency * x[i]) + offset;
        }
    """
    
    # Create the callback generator
    generator = create_callback_generator(transformation_code)
    
    # Generate callbacks for each parameter
    param_list = list(parameters.keys())
    callbacks = {
        param: generator(param, param_list) for param in param_list
    }
    
    # Create the plot
    layout = create_interactive_plot(
        data_df=data,
        x_column='x',
        y_column='y',
        plot_title='Generated Callbacks Demo',
        parameters=parameters,
        callback_functions=callbacks,
        output_filename='generated_callbacks_demo.html'
    )
    
    return layout




import numpy as np
import pandas as pd
from bokeh.plotting import show

# Create sample data
x = np.linspace(0, 10, 100)
y = np.sin(x)
data = pd.DataFrame({'x': x, 'y': y})

# Define a simple transformation
transformations = [{
    'name': 'Scale and Shift',
    'function': lambda x, y, scale=1.0, offset=0.0: y * scale + offset,
    'params': {
        'scale': {
            'type': 'slider',
            'title': 'Scale',
            'value': 1.0,
            'start': 0.1,
            'end': 3.0,
            'step': 0.1
        },
        'offset': {
            'type': 'slider',
            'title': 'Offset',
            'value': 0.0,
            'start': -3.0,
            'end': 3.0,
            'step': 0.1
        }
    }
}]

# Create the interactive plot
layout = auto_interactive_plot(
    data_df=data,
    x_col='x',
    y_col='y',
    transformations=transformations,
    title='My Interactive Plot'
)

# Show the plot
show(layout)

NameError: name 'auto_interactive_plot' is not defined

In [45]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, CustomJSTransform
from bokeh.models.widgets import Slider
from bokeh.layouts import column
from bokeh.events import MouseMove

# Create initial data
x_data = [0, 1, 2, 3]
base_y_data = [1, 2, 3, 4]

# Create a ColumnDataSource with original data and a shift parameter
source = ColumnDataSource(data=dict(
    x=x_data,
    base_y=base_y_data,
    y=base_y_data.copy(),  # Start with unmodified data
    shift=[0]  # Single value to store the current shift
))

# Create a simple figure
p = figure(width=400, height=400, title="Interactive Line Plot with setValue")
line = p.line('x', 'y', source=source, line_width=2)

# Create a slider widget for Y-shift
slider = Slider(start=-5, end=5, value=0, step=0.5, title="Y Shift")

# Define a callback that will update the source data when the slider changes
def update_data(attr, old, new):
    # Get the current shift value from the slider
    shift_value = slider.value
    
    # Update the shift value in the source
    source.data['shift'] = [shift_value]
    
    # Calculate new y values by adding the shift to the base values
    new_y = [y + shift_value for y in source.data['base_y']]
    
    # Update the y values in the source
    source.data['y'] = new_y

# Connect the callback to the slider's value change event
slider.on_change('value', update_data)

# Create a layout with the figure and slider
layout = column(p, slider)

# Show the layout
show(layout)

You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



Standalone HTML file 'interactive_plot_minimal.html' has been created.


In [59]:
import numpy as np
import pandas as pd
from IPython.core.display import HTML

import bokeh
print('Bokeh version:', bokeh.__version__)

from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
from bokeh.io import output_notebook
from bokeh.models import HoverTool
from bokeh.resources import CDN
from bokeh.embed import file_html
output_notebook()

df = pd.DataFrame(np.random.normal(0, 5, (100, 2)), columns=['x','y'])
source = ColumnDataSource(df)
hover = HoverTool(tooltips=[("x", "@x"), ("y", "@y")])
myplot = figure(width=600, height=400, tools='hover,box_zoom,box_select,crosshair,reset')
myplot.circle('x', 'y', size=7, fill_alpha=0.5, source=source)
slider = Slider(start=0.1, end=10, value=7, step=0.1, title="Circle Size")
show(myplot, notebook_handle=True);


Bokeh version: 3.6.3


Loading BokehJS ...

In [56]:
myplot_html = file_html(myplot, CDN)
# this HTML code is very long (~30 K), the cell below doesn't show all the code in NBviewer
print(myplot_html) 

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Bokeh Application</title>
    <style>
      html, body {
        box-sizing: border-box;
        display: flow-root;
        height: 100%;
        margin: 0;
        padding: 0;
      }
    </style>
    <script type="text/javascript" src="static/extensions/panel/bundled/reactiveesm/es-module-shims@^1.10.0/dist/es-module-shims.min.js"></script>
    <script type="text/javascript" src="https://cdn.bokeh.org/bokeh/release/bokeh-3.6.3.min.js"></script>
    <script type="text/javascript">
        Bokeh.set_log_level("info");
    </script>
  </head>
  <body>
    <div id="ceb386c9-0ca4-4c32-8681-5c67fdc07eba" data-root-id="35a8512e-d2d4-4419-a896-7e3ad1b708fe" style="display: contents;"></div>
  
    <script type="application/json" id="cb112e86-227b-404b-9355-71ee0ffbaf9b">
      {"a2069e66-c52d-4e49-94c0-a24fc52b4a9e":{"version":"3.6.3","title":"Bokeh Application","roots":[{"type":"object","name":"Figure","id":"35

In [58]:
from IPython.core.display import HTML
HTML(myplot_html)